In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/Telco-Customer-Churn.csv")

print(df.shape)

(7043, 21)


In [2]:
TARGET = "Churn"

X = df.drop(columns=[TARGET])
y = df[TARGET].map({
    "No": 0,
    "Yes": 1
})

In [3]:
X = X.drop(columns=["customerID"])

In [4]:
print("X shape:", X.shape)
print("\ny distribution:")
print(y.value_counts())

print("\nMissing values:")
print(X.isna().sum()[X.isna().sum() > 0])

X shape: (7043, 19)

y distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Missing values:
Series([], dtype: int64)


In [5]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['SeniorCitizen', 'tenure', 'MonthlyCharges']

Categorical features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_34460\1263442432.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [26]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC


# ==========================================
# NUMERICAL FEATURES
# ==========================================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


# ==========================================
# CATEGORICAL FEATURES
# ==========================================

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()


categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


# ==========================================
# COMBINED PREPROCESSOR
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    sparse_threshold=0
)


# ==========================================
# MODELS
# ==========================================

models = {

    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "HistGradientBoosting": Pipeline([
        ("preprocessor", preprocessor),
        ("model", HistGradientBoostingClassifier(
            random_state=42
        ))
    ]),

    "SVM": Pipeline([
        ("preprocessor", preprocessor),
        ("model", SVC(
            probability=True,
            random_state=42
        ))
    ])
}


print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Models:", list(models.keys()))

Numerical features: 4
Categorical features: 15
Models: ['Logistic Regression', 'Random Forest', 'HistGradientBoosting', 'SVM']


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_34460\55254637.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [27]:
param_grids = {

    "Logistic Regression": {
        "model__C": [0.1, 1, 10]
    },

    "Random Forest": {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 10, 20]
    },

    "HistGradientBoosting": {
        "model__learning_rate": [0.05, 0.1],
        "model__max_iter": [100, 200],
        "model__max_leaf_nodes": [15, 31]
    },

    "SVM": {
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", "auto"]
    }
}

print("Parameter grids ready.")

Parameter grids ready.


In [28]:
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

nested_results = []

for model_name, pipeline in models.items():

    print(f"\nRunning: {model_name}")

    scores = []

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X, y),
        start=1
    ):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        search = GridSearchCV(
            estimator=clone(pipeline),
            param_grid=param_grids[model_name],
            cv=inner_cv,
            scoring="roc_auc",
            n_jobs=1,
            error_score="raise"
        )

        search.fit(X_train, y_train)

        probabilities = search.predict_proba(
            X_test
        )[:, 1]

        score = roc_auc_score(
            y_test,
            probabilities
        )

        scores.append(score)

        print(
            f"  Fold {fold}: "
            f"ROC-AUC = {score:.4f}"
        )

    nested_results.append({
        "Model": model_name,
        "ROC_AUC_Mean": np.mean(scores),
        "ROC_AUC_STD": np.std(scores)
    })

print("\n==============================")
print("NESTED CV COMPLETED")
print("==============================")


Running: Logistic Regression
  Fold 1: ROC-AUC = 0.8544
  Fold 2: ROC-AUC = 0.8455
  Fold 3: ROC-AUC = 0.8631
  Fold 4: ROC-AUC = 0.8253
  Fold 5: ROC-AUC = 0.8366

Running: Random Forest
  Fold 1: ROC-AUC = 0.8567
  Fold 2: ROC-AUC = 0.8385
  Fold 3: ROC-AUC = 0.8562
  Fold 4: ROC-AUC = 0.8239
  Fold 5: ROC-AUC = 0.8369

Running: HistGradientBoosting
  Fold 1: ROC-AUC = 0.8588
  Fold 2: ROC-AUC = 0.8439
  Fold 3: ROC-AUC = 0.8574
  Fold 4: ROC-AUC = 0.8318
  Fold 5: ROC-AUC = 0.8393

Running: SVM


c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning:

  Fold 1: ROC-AUC = 0.8424


c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning:

  Fold 2: ROC-AUC = 0.8316


c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning:

  Fold 3: ROC-AUC = 0.8594


c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\LENOVO\Desktop\ml-eval-framework\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning:

KeyboardInterrupt: 

In [ ]:
results_df.to_csv(
    "../experiments/nested_cv_results.csv",
    index=False
)

print("✅ Nested CV results saved.")